In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import json
import csv
import os
from pathlib import Path

# Фиксация seed для воспроизводимости
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# === ВЫБЕРИ ОДИН ВАРИАНТ ИЗ ТРЁХ ===
DATASET_CHOICE = "CIFAR10"  # "KMNIST", "EMNIST", или "CIFAR10"
DATA_ROOT = "./data"

# Трансформации
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Загрузка датасета
if DATASET_CHOICE == "KMNIST":
    train_full = datasets.KMNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
    test_dataset = datasets.KMNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
    num_classes = 10
elif DATASET_CHOICE == "EMNIST":
    train_full = datasets.EMNIST(root=DATA_ROOT, split="balanced", train=True, download=True, transform=transform)
    test_dataset = datasets.EMNIST(root=DATA_ROOT, split="balanced", train=False, download=True, transform=transform)
    num_classes = 47
elif DATASET_CHOICE == "CIFAR10":
    train_full = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=transform)
    num_classes = 10
else:
    raise ValueError("Выберите KMNIST, EMNIST или CIFAR10")

# Разбиение train/val (80/20)
train_size = int(0.8 * len(train_full))
val_size = len(train_full) - train_size
train_dataset, val_dataset = random_split(train_full, [train_size, val_size], 
                                          generator=torch.Generator().manual_seed(SEED))

# DataLoader'ы
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Sanity check
for x, y in train_loader:
    print(f"Dataset: {DATASET_CHOICE}")
    print(f"Batch shape: x={x.shape}, y={y.shape}")
    print(f"Value range: [{x.min():.3f}, {x.max():.3f}]")
    print(f"Number of classes: {num_classes}")
    print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")
    break

Dataset: CIFAR10
Batch shape: x=torch.Size([64, 3, 32, 32]), y=torch.Size([64])
Value range: [0.000, 1.000]
Number of classes: 10
Train size: 40000, Val size: 10000, Test size: 10000


In [3]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_sizes, num_classes, 
                 use_dropout=False, dropout_p=0.3, use_batchnorm=False):
        super().__init__()
        layers = []
        
        # Первый слой: Flatten
        layers.append(nn.Flatten())
        
        prev_size = input_size
        for h_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, h_size))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h_size))
            layers.append(nn.ReLU())
            if use_dropout:
                layers.append(nn.Dropout(dropout_p))
            prev_size = h_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

def get_input_size(dataset_choice):
    if dataset_choice == "CIFAR10":
        return 3 * 32 * 32  # 3072
    elif dataset_choice in ["MNIST", "KMNIST", "EMNIST"]:
        return 1 * 28 * 28  # 784
    else:
        return 784

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return total_loss / len(loader), correct / total

In [4]:
def run_experiment(model_config, optimizer_config, epochs, early_stopping_patience=None):
    input_size = get_input_size(DATASET_CHOICE)
    model = MLP(
        input_size=input_size,
        hidden_sizes=model_config["hidden_sizes"],
        num_classes=num_classes,
        use_dropout=model_config.get("use_dropout", False),
        dropout_p=model_config.get("dropout_p", 0.3),
        use_batchnorm=model_config.get("use_batchnorm", False)
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    if optimizer_config["type"] == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=optimizer_config["lr"], 
                              weight_decay=optimizer_config.get("weight_decay", 0))
    else:  # SGD
        optimizer = optim.SGD(model.parameters(), lr=optimizer_config["lr"],
                             momentum=optimizer_config.get("momentum", 0),
                             weight_decay=optimizer_config.get("weight_decay", 0))
    
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_acc = evaluate(model, val_loader, criterion, device)
        
        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["train_acc"].append(t_acc)
        history["val_acc"].append(v_acc)
        
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if early_stopping_patience and patience_counter >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, history, best_val_acc, history["val_loss"][-1], len(history["train_loss"])

In [5]:
results = []
artifacts_dir = Path("artifacts")
figures_dir = artifacts_dir / "figures"
artifacts_dir.mkdir(exist_ok=True)
figures_dir.mkdir(exist_ok=True)

# Базовая архитектура
BASE_HIDDEN = [256, 128, 64]

# E1: Base
print("=== E1: Base ===")
model_e1, hist_e1, val_acc_e1, val_loss_e1, epochs_e1 = run_experiment(
    model_config={"hidden_sizes": BASE_HIDDEN},
    optimizer_config={"type": "Adam", "lr": 1e-3},
    epochs=20
)
results.append({
    "experiment_id": "E1", "dataset": DATASET_CHOICE, "seed": SEED,
    "model_summary": f"hidden={BASE_HIDDEN}, act=ReLU",
    "optimizer": "Adam", "lr": 1e-3, "momentum": 0, "weight_decay": 0,
    "epochs_trained": epochs_e1, "best_val_accuracy": val_acc_e1, "best_val_loss": val_loss_e1
})

# E2: +Dropout
print("=== E2: Dropout ===")
model_e2, hist_e2, val_acc_e2, val_loss_e2, epochs_e2 = run_experiment(
    model_config={"hidden_sizes": BASE_HIDDEN, "use_dropout": True, "dropout_p": 0.3},
    optimizer_config={"type": "Adam", "lr": 1e-3},
    epochs=20
)
results.append({
    "experiment_id": "E2", "dataset": DATASET_CHOICE, "seed": SEED,
    "model_summary": f"hidden={BASE_HIDDEN}, act=ReLU, dropout=0.3",
    "optimizer": "Adam", "lr": 1e-3, "momentum": 0, "weight_decay": 0,
    "epochs_trained": epochs_e2, "best_val_accuracy": val_acc_e2, "best_val_loss": val_loss_e2
})

# E3: +BatchNorm
print("=== E3: BatchNorm ===")
model_e3, hist_e3, val_acc_e3, val_loss_e3, epochs_e3 = run_experiment(
    model_config={"hidden_sizes": BASE_HIDDEN, "use_batchnorm": True},
    optimizer_config={"type": "Adam", "lr": 1e-3},
    epochs=20
)
results.append({
    "experiment_id": "E3", "dataset": DATASET_CHOICE, "seed": SEED,
    "model_summary": f"hidden={BASE_HIDDEN}, act=ReLU, BatchNorm",
    "optimizer": "Adam", "lr": 1e-3, "momentum": 0, "weight_decay": 0,
    "epochs_trained": epochs_e3, "best_val_accuracy": val_acc_e3, "best_val_loss": val_loss_e3
})

# E4: EarlyStopping на лучшем из E2/E3
best_reg = "E2" if val_acc_e2 >= val_acc_e3 else "E3"
print(f"=== E4: EarlyStopping on {best_reg} ===")
model_config_e4 = {"hidden_sizes": BASE_HIDDEN, "use_dropout": best_reg=="E2", "dropout_p": 0.3} if best_reg=="E2" else {"hidden_sizes": BASE_HIDDEN, "use_batchnorm": True}
model_e4, hist_e4, val_acc_e4, val_loss_e4, epochs_e4 = run_experiment(
    model_config=model_config_e4,
    optimizer_config={"type": "Adam", "lr": 1e-3},
    epochs=50,
    early_stopping_patience=5
)
results.append({
    "experiment_id": "E4", "dataset": DATASET_CHOICE, "seed": SEED,
    "model_summary": f"{model_config_e4}, EarlyStopping(patience=5)",
    "optimizer": "Adam", "lr": 1e-3, "momentum": 0, "weight_decay": 0,
    "epochs_trained": epochs_e4, "best_val_accuracy": val_acc_e4, "best_val_loss": val_loss_e4
})

# Сохранение лучшей модели (E4)
torch.save(model_e4.state_dict(), artifacts_dir / "best_model.pt")

# График для E4
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_e4["train_loss"], label="train loss")
plt.plot(hist_e4["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("E4: Loss curves")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(hist_e4["train_acc"], label="train acc")
plt.plot(hist_e4["val_acc"], label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("E4: Accuracy curves")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(figures_dir / "curves_best.png", dpi=150)
plt.close()

=== E1: Base ===
=== E2: Dropout ===
=== E3: BatchNorm ===
=== E4: EarlyStopping on E3 ===
Early stopping at epoch 12


In [6]:
# Фиксированная архитектура как в E4
fixed_model_config = model_config_e4

# O1: LR too large
print("=== O1: LR too large ===")
_, hist_o1, _, _, _ = run_experiment(
    model_config=fixed_model_config,
    optimizer_config={"type": "Adam", "lr": 1e-1},
    epochs=8
)
results.append({
    "experiment_id": "O1", "dataset": DATASET_CHOICE, "seed": SEED,
    "model_summary": str(fixed_model_config),
    "optimizer": "Adam", "lr": 1e-1, "momentum": 0, "weight_decay": 0,
    "epochs_trained": 8, 
    "best_val_accuracy": max(hist_o1["val_acc"]),  # ИСПРАВЛЕНО: лучшая accuracy
    "best_val_loss": min(hist_o1["val_loss"])       # ИСПРАВЛЕНО: лучший loss
})

# O2: LR too small
print("=== O2: LR too small ===")
_, hist_o2, _, _, _ = run_experiment(
    model_config=fixed_model_config,
    optimizer_config={"type": "Adam", "lr": 1e-5},
    epochs=8
)
results.append({
    "experiment_id": "O2", "dataset": DATASET_CHOICE, "seed": SEED,
    "model_summary": str(fixed_model_config),
    "optimizer": "Adam", "lr": 1e-5, "momentum": 0, "weight_decay": 0,
    "epochs_trained": 8,
    "best_val_accuracy": max(hist_o2["val_acc"]),  # ИСПРАВЛЕНО: лучшая accuracy
    "best_val_loss": min(hist_o2["val_loss"])       # ИСПРАВЛЕНО: лучший loss
})

# O3: SGD+momentum+weight_decay
print("=== O3: SGD+momentum+wd ===")
_, hist_o3, val_acc_o3, val_loss_o3, epochs_o3 = run_experiment(
    model_config=fixed_model_config,
    optimizer_config={"type": "SGD", "lr": 1e-2, "momentum": 0.9, "weight_decay": 1e-4},
    epochs=15
)
results.append({
    "experiment_id": "O3", "dataset": DATASET_CHOICE, "seed": SEED,
    "model_summary": str(fixed_model_config),
    "optimizer": "SGD", "lr": 1e-2, "momentum": 0.9, "weight_decay": 1e-4,
    "epochs_trained": epochs_o3, "best_val_accuracy": val_acc_o3, "best_val_loss": val_loss_o3
})

# График O1 vs O2
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(hist_o1["train_loss"], label="O1 train (lr=1e-1)", alpha=0.7)
plt.plot(hist_o1["val_loss"], label="O1 val (lr=1e-1)", alpha=0.7)
plt.plot(hist_o2["train_loss"], label="O2 train (lr=1e-5)", alpha=0.7)
plt.plot(hist_o2["val_loss"], label="O2 val (lr=1e-5)", alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LR extremes comparison")
plt.legend(fontsize=8)
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(hist_o1["val_acc"], label="O1 val acc (lr=1e-1)", alpha=0.7)
plt.plot(hist_o2["val_acc"], label="O2 val acc (lr=1e-5)", alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("Val Accuracy")
plt.title("Validation Accuracy")
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.savefig(figures_dir / "curves_lr_extremes.png", dpi=150)
plt.close()

=== O1: LR too large ===
=== O2: LR too small ===
=== O3: SGD+momentum+wd ===


In [7]:
# Проверка существования модели
if 'model_e4' not in locals():
    raise RuntimeError("Сначала выполните Ячейку 5 (эксперименты E1-E4)!")

# Финальная оценка лучшей модели (E4) на test
test_loss, test_acc = evaluate(model_e4, test_loader, nn.CrossEntropyLoss(), device)
print(f"Final test accuracy (best model E4): {test_acc:.4f}")

# Сохранение runs.csv
with open(artifacts_dir / "runs.csv", "w", newline="", encoding="utf-8") as f:
    fieldnames = ["experiment_id", "dataset", "seed", "model_summary", "optimizer", 
                  "lr", "momentum", "weight_decay", "epochs_trained", "best_val_accuracy", "best_val_loss"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow(r)

# Сохранение best_config.json
best_config = {
    "dataset": DATASET_CHOICE,
    "seed": SEED,
    "model_architecture": fixed_model_config,
    "optimizer": "Adam",
    "lr": 1e-3,
    "early_stopping_patience": 5,
    "final_test_accuracy": test_acc
}
with open(artifacts_dir / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, indent=2, ensure_ascii=False)

print("✓ Все артефакты сохранены в artifacts/")
print(f"✓ Final test accuracy: {test_acc:.4f}")

Final test accuracy (best model E4): 0.5032
✓ Все артефакты сохранены в artifacts/
✓ Final test accuracy: 0.5032
